# Service Discovery

CoreDNS löst Service-Namen innerhalb des Clusters auf. Ein regulärer Service-Name verweist auf seine ClusterIP. Gleichnamige Services in verschiedenen Namespaces sind unterscheidbar.

![namespace-dns](namespace-dns.svg)

`ent.prod.svc.cluster.local`: Service ent, Namespace prod, Service-DNS-Zone svc, Cluster-Domain cluster.local (hier K3s-Standard).

## Voraussetzungen

Python-Kernel mit %%bash, kubectl und Zugriff auf den K3s-Trainingscluster. Notebook aus dem Kapitelverzeichnis ausführen. Beispielressourcen müssen frei sein; vorherige Übungen aufräumen. Aktiver Namespace muss default sein.

In [ ]:
%%bash
set -eo pipefail
kubectl config current-context
NS=$(kubectl config view --minify -o jsonpath='{..namespace}')
test "${NS:-default}" = default
kubectl get nodes

## 1. Zwei Namespaces

dev und prod müssen neu sein; die Prüfung verhindert versehentliches Löschen vorhandener Ressourcen.

In [ ]:
%%bash
set -eo pipefail
for NS in dev prod; do
    if kubectl get namespace "$NS" --ignore-not-found -o name | grep -q .; then echo "Namespace $NS existiert bereits"; exit 1; fi
done
cat sd-example.yml

In [ ]:
%%bash
set -eo pipefail
kubectl apply -f sd-example.yml
kubectl -n dev rollout status deployment/enterprise --timeout=180s
kubectl -n prod rollout status deployment/enterprise --timeout=180s
kubectl -n dev wait --for=condition=Ready pod/jump --timeout=180s
kubectl get svc -n dev
kubectl get svc -n prod

## 2. DNS Search Domains

resolv.conf enthält Nameserver, Search Domains und Optionen wie ndots. Kurze Namen werden durch Search Domains ergänzt; im Namespace dev verweist ent auf dev.

In [ ]:
%%bash
set -eo pipefail
kubectl -n dev exec jump -- cat /etc/resolv.conf
kubectl -n dev exec jump -- nslookup ent
kubectl -n dev exec jump -- nslookup ent.prod.svc.cluster.local

## 3. HTTP-Zugriff

ent und ent.dev liefern Development, ent.prod und der vollständige Name liefern Production.

In [ ]:
%%bash
set -eo pipefail
for NAME in ent ent.dev ent.prod ent.prod.svc.cluster.local; do
    echo "$NAME"
    kubectl -n dev exec jump -- wget -qO- "http://$NAME:8080/"
    echo
done

## 4. Nicht existierender Name

NXDOMAIN bedeutet, dass der angefragte DNS-Name nicht existiert.

In [ ]:
%%bash
set -eo pipefail
kubectl -n dev exec jump -- nslookup does-not-exist.dev.svc.cluster.local || echo 'Namensauflösung erwartungsgemäß fehlgeschlagen'

## 5. Aufräumen

Nur die eigens für diese Übung erzeugten Namespaces löschen.

In [ ]:
%%bash
set -eo pipefail
kubectl delete namespace dev prod
kubectl get namespace dev prod --ignore-not-found

## Fazit

DNS-Namen ermöglichen stabile, Namespace-übergreifende Service Discovery ohne Kenntnis der Pod-IPs.